In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.simulation.order_stream import OrderStream
from src.simulation.simulator import Simulator
from src.dynamicProgramming.dp_scheduler import DPScheduler
from src.dynamicProgramming.insertion_policy import GreedyInsertionPolicy

from geopy.distance import geodesic

data_path = Path("../data/features/olist/delivery_jobs_dataset.csv")

In [ ]:
# creating a synthetic dispatching scenario
jobs = pd.read_csv(
    data_path,
    parse_dates = [
        'ready_time',
        'due_date'
    ]
)

# removing rows with missing ccoordinates
jobs = jobs.dropna(
    subset=[
        'pickup_lat',
        'pickup_lng',
        'delivery_lat',
        'delivery_lng'
    ]
)

def distance(row):
    return geodesic(
        (row.pickup_lat, row.pickup_lng),
        (row.delivery_lat, row.delivery_lng)
    ).km

jobs['distance_km'] = jobs.apply(distance, axis = 1)

jobs = jobs[jobs['distance_km'] < 15]

jobs = jobs.sort_values('ready_time').head(10).copy()

start = pd.Timestamp("2024-01-01 08:00:00")

jobs['ready_time'] = start + pd.to_timedelta(
    np.arange(len(jobs)) * 5,
    unit = 'm'
)

jobs['due_date'] = jobs['ready_time'] + pd.Timedelta(hours = 4)

jobs['service_time_min'] = 5

display(jobs)

# creating instances for Simulation setup
stream = OrderStream(jobs)
policy = GreedyInsertionPolicy(regularization_lambda=2.0)
scheduler = DPScheduler(
    insertion_policy = policy,
    n_couriers = 5
)
simulator = Simulator(stream, scheduler)

state = simulator.run(
    start_time = jobs['ready_time'].min(),
    end_time = jobs['ready_time'].min() + pd.Timedelta(hours = 6),
    step_minutes = 5
)

for courier in state.couriers:
    print(f"Courier {courier.courier_id}")
    print(f"Route length: {len(courier.route)}")
    print(f"Completed jobs: {len(courier.completed_jobs)}")
    print(f"Completed job IDs: {[job['job_id'] for job in courier.completed_jobs]}")
    print(f"Current route job IDs: {[job['job_id'] for job in courier.route]}")
    print("---")

,job_id,order_id,seller_id,customer_id,pickup_lat,pickup_lng,delivery_lat,delivery_lng,ready_time,due_date,service_time_min,demand,distance_km
66586,ed8c7b1b3eb256c70ce0c74231e1da88,ed8c7b1b3eb256c70ce0c74231e1da88,5b179e9e8cc7ab6fd113a46ca584da81,da0ba2a9935bca5b4610b0e3bca9d3b4,-23.568771,-46.698110,-23.453962,-46.731884,2024-01-01 08:00:00,2024-01-01 12:00:00,5,1.0,13.174828
31857,79ffdd52a918bbe867895a4b183d6457,79ffdd52a918bbe867895a4b183d6457,c7dcd301ecfe5ab7f778ac172cf74be7,f9808148a262b51d20e2d777eee6676c,-19.918527,-43.939890,-19.948076,-43.947618,2024-01-01 08:05:00,2024-01-01 12:05:00,5,1.0,3.369796
16233,c4b41c36dd589e901f6879f25a74ec1d,c4b41c36dd589e901f6879f25a74ec1d,ce27a3cc3c8cc1ea79d11e561e9bebb6,4bb880cac21c7a9e1371ab1ebd601706,-23.541812,-46.624550,-23.534776,-46.671765,2024-01-01 08:10:00,2024-01-01 12:10:00,5,1.0,4.883740
93439,6b3ee7697a02619a0ace2b3f0aa46bde,6b3ee7697a02619a0ace2b3f0aa46bde,ce27a3cc3c8cc1ea79d11e561e9bebb6,21a6abdf0197fbe57451bd0a1d3c59a2,-23.541812,-46.624550,-23.521920,-46.482302,2024-01-01 08:15:00,2024-01-01 12:15:00,5,1.0,14.692025
3781,3b2ca3293a7ce539ea2379d704fa37ce,3b2ca3293a7ce539ea2379d704fa37ce,dd2bdf855a9172734fbc3744021ae9b9,06a70917afd2dcd59396e1eac836c646,-19.869495,-43.950944,-19.843643,-43.904064,2024-01-01 08:20:00,2024-01-01 12:20:00,5,1.0,5.683464
31851,b2f92b2f7047cd8b35580d629d7b3bfb,b2f92b2f7047cd8b35580d629d7b3bfb,ecccfa2bb93b34a3bf033cc5d1dcdc69,1cbeb91a58bd89ee2df1bc32f3311209,-25.507014,-49.275963,-25.512877,-49.183727,2024-01-01 08:25:00,2024-01-01 12:25:00,5,1.0,9.295196
13131,98974b076b01553d49ee6467905675a7,98974b076b01553d49ee6467905675a7,5b179e9e8cc7ab6fd113a46ca584da81,dc7cdb748679fb6f280f66d7582c5e59,-23.568771,-46.698110,-23.600389,-46.644506,2024-01-01 08:30:00,2024-01-01 12:30:00,5,1.0,6.496278
79804,59e7ca596664da0a459a145eb95cc9c5,59e7ca596664da0a459a145eb95cc9c5,46dc3b2cc0980fb8ec44634e21d2718e,c851e38392f26d6a33bfe6f717c0fba8,-22.935263,-43.187264,-22.915669,-43.225213,2024-01-01 08:35:00,2024-01-01 12:35:00,5,1.0,4.456774
7429,323dad8c483c7f8b818a825d257f4aa0,323dad8c483c7f8b818a825d257f4aa0,2138ccb85b11a4ec1e37afbd1c8eda1f,021833e9737a1145b5a6e3053f0f4329,-23.539944,-46.439816,-23.601092,-46.553696,2024-01-01 08:40:00,2024-01-01 12:40:00,5,1.0,13.454306
91786,f7310018040436b01ab03f81b301b5de,f7310018040436b01ab03f81b301b5de,5b179e9e8cc7ab6fd113a46ca584da81,efec125e0fd56164c37c9f1e1c804cd7,-23.568771,-46.698110,-23.558371,-46.582149,2024-01-01 08:45:00,2024-01-01 12:45:00,5,3.0,11.894541


Courier 0 current route cost: 5.0
Courier 1 current route cost: 5.0
Courier 2 current route cost: 5.0
Courier 3 current route cost: 5.0
Courier 4 current route cost: 5.0
Trying job: ed8c7b1b3eb256c70ce0c74231e1da88
Feasible route found
Assigning job ed8c7b1b3eb256c70ce0c74231e1da88 to courier 0 at position 0 with cost 5.0
Courier 0 current route cost: 5.0
Courier 1 current route cost: 5.0
Courier 2 current route cost: 5.0
Courier 3 current route cost: 5.0
Courier 4 current route cost: 5.0
Trying job: 79ffdd52a918bbe867895a4b183d6457
Feasible route found
Assigning job 79ffdd52a918bbe867895a4b183d6457 to courier 0 at position 0 with cost 5.0
Courier 0 current route cost: 5.0
Courier 1 current route cost: 5.0
Courier 2 current route cost: 5.0
Courier 3 current route cost: 5.0
Courier 4 current route cost: 5.0
Trying job: c4b41c36dd589e901f6879f25a74ec1d
Feasible route found
Assigning job c4b41c36dd589e901f6879f25a74ec1d to courier 0 at position 0 with cost 5.0
Courier 0 current route cost